# Introduction to Large Language Models (LLMs) and Pre-training

This notebook illustrates the concepts of different LLM architectures and their pre-training objectives:

Masked Language Modeling (MLM) with BERT (Encoder-only)

Semantic Similarity with Sentence-BERT

Causal Language Modeling (CLM) with GPT (Decoder-only)

Sequence-to-Sequence (Seq2Seq) with BART / T5 (Encoder-Decoder)

## 0. Install Dependencies

In [1]:
# Install the necessary libraries
!pip install transformers torch sentence-transformers

## 1. BERT (Bidirectional Encoder Representations from Transformers)

Architecture: Encoder-Only

Pre-training Objective: Masked Language Modeling (MLM)

BERT is pre-trained to guess a masked word in a sentence (randomly masking 15% of tokens) using bidirectional context.

In [3]:
from transformers import pipeline

# Initialize the MLM pipeline with a BERT model
mlm_pipeline = pipeline("fill-mask", model="bert-base-uncased")

# Sentence with a masked token
text = "Apple presented the new [MASK] X yesterday during its conference."

print(f"Original sentence: {text}\n")
results = mlm_pipeline(text)

print("Predictions for the masked word:")
for res in results:
    print(f"- {res['token_str']} (Score: {res['score']:.4f})")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Original sentence: Apple presented the new [MASK] X yesterday during its conference.

Predictions for the masked word:
- os (Score: 0.4509)
- iphone (Score: 0.1469)
- apple (Score: 0.1007)
- generation (Score: 0.0301)
- model (Score: 0.0205)


## 2. Sentence-BERT (SBERT)

SBERT modifies the BERT architecture (using siamese networks) to derive semantically meaningful sentence embeddings that can be compared using cosine similarity.

In [4]:
from sentence_transformers import SentenceTransformer, util

# Load the model (identical to the course example)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode two sentences
emb1 = model.encode("This is a red cat with a hat.")
emb2 = model.encode("Have you seen my red cat?")

# Compute cosine similarity
cos_sim = util.cos_sim(emb1, emb2)

print(f"Sentence 1: This is a red cat with a hat.")
print(f"Sentence 2: Have you seen my red cat?")
print(f"Cosine Similarity: {cos_sim.item():.4f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence 1: This is a red cat with a hat.
Sentence 2: Have you seen my red cat?
Cosine Similarity: 0.6153


## 3. GPT (Generative Pre-trained Transformer)

Architecture: Decoder-Only

Pre-training Objective: Causal Language Modeling (CLM)

GPT generates text in an autoregressive manner. It predicts the next token in a sequence by only looking at the previous tokens (left context).

In [5]:
# Initialize the text generation pipeline with GPT-2
gpt_pipeline = pipeline("text-generation", model="gpt2")

prompt = "The computer which I had just put into the machine room"

print(f"Prompt: {prompt}")
# Generate the rest of the text
generated_text = gpt_pipeline(prompt, max_length=50, num_return_sequences=1, truncation=True)

print("\nGenerated text:")
print(generated_text[0]['generated_text'])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: The computer which I had just put into the machine room


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Generated text:
The computer which I had just put into the machine room was the same one I had plugged into my computer at home. It was a single computer which was connected to a computer in a room full of computers.

I told the computer operator that it could take up to 10 minutes for my computer to start up. He said to me, "You can't do that. You have to get a computer that is 100 percent up and running."

I said, "Okay, let's go to the computer."

A computer operator looked at me seriously and said, "You need to shut this down."

I said, "Go ahead."

The computer operator said, "You can't shut it down. You have to shut it down."

I said, "Okay, so I can't shut it down that computer."

He said, "I can't do that."

I said, "Okay, I can't do that."

I had been working on a computer for a year. I was working on a computer which was running at full power. It was running at full power.

The computer operator said, "You can shut it down. You have to shut it down."

I said, "Okay, you can 

## 4. BART / T5 and the Encoder-Decoder Architecture

Architecture: Encoder-Decoder

Pre-training Objective: Denoising Autoencoder / Text-to-Text

These models use a bidirectional encoder (like BERT) and an autoregressive decoder (like GPT). They are excellent for conditional generation tasks like text summarization or translation.

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# A long text to summarize
long_text = """
Attention is a powerful mechanism to create context-aware representations.
A model that follows the encoder-decoder structure maps input tokens to a sequence of continuous representations,
which are then consumed by the decoder to generate a sequence of outputs.
The Transformer relies entirely on self-attention mechanisms, dispensing with recurrence and convolutions completely.
This allows the model to directly "look" at all tokens and decide which one is useful, solving the long-range dependencies problem.
"""

print("Original text:")
print(long_text)

inputs = tokenizer(long_text, return_tensors="pt", max_length=1024, truncation=True)

summary_ids = model.generate(inputs["input_ids"], max_length=40, min_length=15, do_sample=False)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("\nSummary generated by BART:")
print(summary)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Original text:

Attention is a powerful mechanism to create context-aware representations. 
A model that follows the encoder-decoder structure maps input tokens to a sequence of continuous representations, 
which are then consumed by the decoder to generate a sequence of outputs. 
The Transformer relies entirely on self-attention mechanisms, dispensing with recurrence and convolutions completely.
This allows the model to directly "look" at all tokens and decide which one is useful, solving the long-range dependencies problem.


Summary generated by BART:
Attention is a powerful mechanism to create context-aware representations. A model that follows the encoder-decoder structure maps input tokens to a sequence of continuous representations.
